In [18]:
# asyncio is python built-in library

# 1. Basic Coroutine
# A coroutine is a specialized version of a
# Python function that can pause its execution
# and yield control back to the "event loop,"
# allowing other tasks to run in the meantime.


# Async Function: You call it, and it returns a coroutine object
# (a wrapper around the function). It hasn't actually "run"
# the code inside yet; it’s just waiting for an event loop to start it.

# await: The Yield Point
# The await keyword is the most powerful part of the duo.
# It can only be used inside an async function.
# When Python hits an await line, it does three things:
# Pauses the current coroutine.
# Yields control back to the event loop.
# Tells the loop: "I'm waiting for this specific thing to finish.
# Go run other code in the meantime, and wake me up when this is done."

# async	Defines a function as a "pausable" task.
# Placing an order.
# await	Pauses execution to wait for a result without
# freezing the program.


import asyncio


# a coroutine. Calling greet() doesn't actually run the code;
# it creates a "coroutine object" that is ready to run.
async def greet():
    print("Hello")
    await asyncio.sleep(1)
    print("World")


# asyncio.run(greet())
# await: This is the "pause" button.
# It tells the event loop: "I'm going to be waiting for 1 second.
# Go ahead and run other code while I'm idle."
# Coroutine Can pause and resume at await points.
# "Non-blocking"—allows other tasks to run during waits.
await greet()  # in jupyter

Hello
World


In [14]:
# 2. Multiple coroutines with gather

import asyncio


async def fetch(name, delay):
    await asyncio.sleep(delay)
    return f"{name} done"


# Return a future aggregating results from the given coroutines/futures.
async def main():
    results = await asyncio.gather(fetch("A", 1), fetch("B", 2), fetch("C", 3))
    print(results)


# asyncio.run(main())
await main()

['A done', 'B done', 'C done']


In [24]:
# Task creation with create_task

import asyncio


async def worker(n):
    await asyncio.sleep(n)
    return f"Worker {n} finished"


async def main():
    tasks = [asyncio.create_task(worker(i)) for i in range(5)]
    # await task one by one
    # for task in tasks:
    #     print(await task)

    # This waits for all 5 to finish and returns a list of results
    # if you want to wait for all of them to finish and get the results at once:
    results = await asyncio.gather(*tasks)
    print(results)


await main()

['Worker 0 finished', 'Worker 1 finished', 'Worker 2 finished', 'Worker 3 finished', 'Worker 4 finished']


In [27]:
# 4. return_await vs return

import asyncio


async def inner():
    await asyncio.sleep(0.1)
    return "inner result"


# Bad: add extra coroutine wrapper
async def bad_wrapper():
    return inner()


# Good: delegates directly
async def good_wrapper():
    return await inner()


async def main():
    print(await good_wrapper())


await main()

inner result


In [29]:
# 5. Timeout with wait_for

import asyncio


async def slow_operation():
    await asyncio.sleep(3)
    return "Completed"


async def main():
    try:
        result = await asyncio.wait_for(slow_operation(), timeout=1.0)
    except asyncio.TimeoutError:
        print("Operation timed out!")


await main()

Operation timed out!


In [33]:
# 6. Shield from cancellatio

import asyncio


async def critical_task():
    await asyncio.sleep(2)
    return "Critical data"


async def main():
    try:
        result = await asyncio.wait_for(asyncio.shield(critical_task()), timeout=1.0)
    except asyncio.TimeoutError:
        print("Timeout but task continues in background")


await main()

Timeout but task continues in background


In [35]:
# 7. wait with FIRST_COMPLETED

import asyncio


async def fast():
    await asyncio.sleep(0.1)
    return "fast"


async def slow():
    await asyncio.sleep(2)
    return "slow"


async def main():
    done, pending = await asyncio.wait(
        [asyncio.create_task(fast()), asyncio.create_task(slow())],
        return_when=asyncio.FIRST_COMPLETED,
    )
    print(done, pending)
    for task in done:
        print(f"Done: {task.result()}")
    for task in pending:
        task.cancel()


await main()

{<Task finished name='Task-61' coro=<fast() done, defined at /var/folders/l1/dzcw8wr90wxdt447b9wlfw3r0000gn/T/ipykernel_77646/2373859862.py:6> result='fast'>} {<Task pending name='Task-62' coro=<slow() running at /var/folders/l1/dzcw8wr90wxdt447b9wlfw3r0000gn/T/ipykernel_77646/2373859862.py:12> wait_for=<Future pending cb=[Task.task_wakeup()]>>}
Done: fast


In [36]:
# 8. wait with FIRST_EXCEPTION

import asyncio


async def failing():
    await asyncio.sleep(0.1)
    raise ValueError("Error occurred")


async def passing():
    await asyncio.sleep(2)
    return "success"


async def main():
    done, pending = await asyncio.wait(
        [asyncio.create_task(failing()), asyncio.create_task(passing())],
        return_when=asyncio.FIRST_EXCEPTION,
    )
    for task in done:
        try:
            print(f"Result: {task.result()}")
        except Exception as e:
            print(f"Exception: {e}")
    for task in pending:
        task.cancel()


await main()

Exception: Error occurred


In [39]:
# as_completed iterator

import asyncio


async def fetch_item(n):
    await asyncio.sleep(n * 0.5)
    return f"item {n}"


async def main():
    tasks = [asyncio.create_task(fetch_item(i)) for i in range(5, 0, -1)]
    # for coro in asyncio.as_completed(tasks):
    #     print(await coro)
    results = await asyncio.gather(*tasks)
    print(results)


await main()

['item 5', 'item 4', 'item 3', 'item 2', 'item 1']


In [43]:
# 10. Semaphore for Rate Limiting

import asyncio

# Limit how many tasks to run at same time
semaphore = asyncio.Semaphore(3)


async def limited_task(name):
    async with semaphore:
        print(f"{name} started")
        await asyncio.sleep(1)
        print(f"{name} finished")


async def main():
    await asyncio.gather(*[limited_task(f"Task-{i}") for i in range(5)])


await main()

Task-0 started
Task-1 started
Task-2 started
Task-0 finished
Task-1 finished
Task-2 finished
Task-3 started
Task-4 started
Task-3 finished
Task-4 finished


In [47]:
# 11. Boundd Semaphore

import asyncio

sem = asyncio.BoundedSemaphore(3)


async def worker(n):
    async with sem:
        print(f"Worker {n} acquired")
        await asyncio.sleep(1)


async def main():
    await asyncio.gather(*[worker(i) for i in range(6)])


await main()

Worker 0 acquired
Worker 1 acquired
Worker 2 acquired
Worker 3 acquired
Worker 4 acquired
Worker 5 acquired


In [50]:
# 12. Lock for mutual exclusion

import asyncio

lock = asyncio.Lock()
counter = 0


async def increment():
    global counter
    async with lock:
        temp = counter
        await asyncio.sleep(0)
        counter = temp + 1
        # print(counter)


async def main():
    await asyncio.gather(*[increment() for _ in range(100)])
    print(f"Counter: {counter}")


await main()

Counter: 100


In [51]:
# 13. Event for signaling

import asyncio

event = asyncio.Event()


async def waiter(name):
    print(f"{name} waiting...")
    await event.wait()
    print(f"{name} triggered!")


async def setter():
    await asyncio.sleep(1)
    print("Setting event")
    event.set()


async def main():
    await asyncio.gather(waiter("A"), waiter("B"), setter())


await main()

A waiting...
B waiting...
Setting event
A triggered!
B triggered!


In [57]:
# 14. Condition for complex synchronization

import asyncio

condition = asyncio.Condition()
items = []


async def producer():
    async with condition:
        items.append("product")
        items.append("product2")
        print("Produced item")
        condition.notify()


async def consumer():
    async with condition:
        await condition.wait_for(lambda: len(items) > 0)
        item = items.pop()
        # item = items.pop()
        print(f"Consumed: {item}")


async def main():
    await asyncio.gather(consumer(), producer())


await main()

Produced item
Consumed: product2


In [63]:
# Simple queue

import asyncio


async def producer(queue):
    for i in range(5):
        await queue.put(i)
        print(f"Produced: {i}")
    await queue.put(None)  # Sentinel


async def consumer(queue):
    while True:
        item = await queue.get()
        if item is None:
            break
        print(f"Consumed: {item}")
        queue.task_done()


async def main():
    queue = asyncio.Queue()
    await asyncio.gather(producer(queue), consumer(queue))


await main()

Produced: 0
Produced: 1
Produced: 2
Produced: 3
Produced: 4
Consumed: 0
Consumed: 1
Consumed: 2
Consumed: 3
Consumed: 4


In [64]:
# 16. Bounded Queue (Backpressure)

import asyncio


async def producer(queue):
    for i in range(5):
        await queue.put(i)
        print(f"Produced: {i}")


async def consumer(queue):
    for _ in range(5):
        item = await queue.get()
        await asyncio.sleep(0.5)  # Slow consumer
        print(f"Consumed: {item}")
        queue.task_done()


async def main():
    queue = asyncio.Queue(maxsize=2)
    await asyncio.gather(producer(queue), consumer(queue))


await main()

Produced: 0
Produced: 1
Produced: 2
Consumed: 0
Produced: 3
Consumed: 1
Produced: 4
Consumed: 2
Consumed: 3
Consumed: 4


In [66]:
# 17. Priority Queue

import asyncio


async def worker(queue):
    while not queue.empty():
        priority, item = await queue.get()
        print(f"Processing priority {priority}: {item}")
        queue.task_done()


async def main():
    queue = asyncio.PriorityQueue()
    await queue.put((3, "low"))
    await queue.put((1, "high"))
    await queue.put((2, "medium"))
    await queue.put((5, "high-medium"))
    await queue.put((6, "high-high"))
    await worker(queue)


await main()

Processing priority 1: high
Processing priority 2: medium
Processing priority 3: low
Processing priority 5: high-medium
Processing priority 6: high-high


In [71]:
# 18. LifoQueue (Stack)

import asyncio


async def main():
    # A subclass of Queue that retrieves most recently added entries first.
    queue = asyncio.LifoQueue()
    for i in range(5):
        await queue.put(i)
        print(f"put {i}")
    while not queue.empty():
        print(f"get {i}")
        print(await queue.get())


await main()

put 0
put 1
put 2
put 3
put 4
get 4
4
get 4
3
get 4
2
get 4
1
get 4
0


In [72]:
# 19.  Barrier for Phase Synchronization

import asyncio


async def phase(name, barrier):
    print(f"{name} phase 1")
    await barrier.wait()
    print(f"{name} phase 2")
    await barrier.wait()
    print(f"{name} phase 3")


async def main():
    barrier = asyncio.Barrier(3)
    await asyncio.gather(*[phase(f"Worker-{i}", barrier) for i in range(3)])


await main()

Worker-0 phase 1
Worker-1 phase 1
Worker-2 phase 1
Worker-2 phase 2
Worker-0 phase 2
Worker-1 phase 2
Worker-0 phase 3
Worker-1 phase 3
Worker-2 phase 3


In [75]:
# 20. Context Manager with Async

import asyncio


class AsyncResource:
    async def __aenter__(self):
        print("Acquiring resource")
        await asyncio.sleep(0.1)
        return self

    async def __aexit__(self, exc_type, exc_val, exc_tab):
        print("Releasing resource")
        await asyncio.sleep(0.1)
        return False

    async def do_work(self):
        print("Working...")
        return "result"


async def main():
    async with AsyncResource() as resource:
        result = await resource.do_work()
        print(f"Got: {result}")


await main()

Acquiring resource
Working...
Got: result
Releasing resource


In [79]:
# 21. Async Iterator

import asyncio


class AsyncRange:
    def __init__(self, stop):
        self.stop = stop
        self.current = 0

    def __aiter__(self):
        return self

    async def __anext__(self):
        if self.current >= self.stop:
            raise StopAsyncIteration
        await asyncio.sleep(0.1)
        value = self.current
        self.current += 1
        return value


async def main():
    async for i in AsyncRange(5):
        print(i)


await main()

0
1
2
3
4


In [82]:
# 22. Async Generator

import asyncio


async def async_counter(n):
    for i in range(n):
        await asyncio.sleep(0.1)
        yield i


async def main():
    async for value in async_counter(5):
        print(value)


await main()

0
1
2
3
4


In [83]:
# 23. async for with enumerate


async def items():
    for i in ["a", "b", "c"]:
        await asyncio.sleep(0.1)
        yield i


async def main():
    async for idx, item in aioenumerate(items()):
        print(f"{idx}: {item}")


async def aioenumerate(aiterable, start=0):
    idx = start
    async for item in aiterable:
        yield idx, item
        idx += 1


await main()

0: a
1: b
2: c


In [85]:
# 24. Cancellation Handling

import asyncio


async def cancellable_task():
    try:
        print("Task started")
        await asyncio.sleep(10)
        print("Task completed")
    except asyncio.CancelledError:
        print("Task was cancelled!")
        raise  # Re-raise for proper cleanup


async def main():
    task = asyncio.create_task(cancellable_task())
    await asyncio.sleep(0.5)
    task.cancel()
    try:
        await task
    except asyncio.CancelledError:
        print("Main caught cancellation")


await main()

Task started
Task was cancelled!
Main caught cancellation


In [88]:
# 25. Cleanup with finally Block

import asyncio


async def task_with_cleanup():
    resource = None
    try:
        print("Acquiring resource")
        resource = "expense-resource"
        await asyncio.sleep(5)
        print("Task done")
    except asyncio.CancelledError:
        print("Cancelled")
        raise
    finally:
        if resource:
            print(f"Cleaning up: {resource}")


async def main():
    task = asyncio.create_task(task_with_cleanup())
    await asyncio.sleep(0.5)
    task.cancel()
    try:
        await task
    except asyncio.CancelledError:
        pass


await main()

Acquiring resource
Cancelled
Cleaning up: expense-resource


In [92]:
# 26. Suppress Cancellation (shidle pattern)

import asyncio


async def cleanup():
    print("Starting cleanup...")
    await asyncio.sleep(1)
    print("Cleanup complete")


async def worker():
    try:
        await asyncio.sleep(10)
    finally:
        # Suppress cancellation during cleanup
        current = asyncio.current_task()
        current.uncancel()
        await cleanup()


async def main():
    task = asyncio.create_task(worker())
    await asyncio.sleep(0.5)
    task.cancel()

    try:
        await task  # The CancelledError is thrown here
    except asyncio.CancelledError:
        print("Main: Task cancellation acknowledged.")


await main()

Starting cleanup...
Cleanup complete
Main: Task cancellation acknowledged.


In [94]:
# 27. Exception Handling in gather

import asyncio


async def failing():
    raise ValueError("Task failed")


async def succeeding():
    await asyncio.sleep(0.1)
    return "success"


async def main():
    # return_exceptions = True collects exceptions instead of raising
    results = await asyncio.gather(failing(), succeeding(), return_exceptions=True)
    for result in results:
        if isinstance(result, Exception):
            print(f"Error: {result}")
        else:
            print(f"Result: {result}")


await main()

Error: Task failed
Result: success


In [96]:
# 28. TaskGroup (Python 3.11+)
import asyncio


async def task(name):
    await asyncio.sleep(0.1)
    return f"{name} done"


async def main():
    results = []
    async with asyncio.TaskGroup() as tg:
        for i in range(3):
            tg.create_task(task(f"Task-{i}"), name=f"task={i}")
    print("All tasks completed")


await main()

All tasks completed


In [97]:
# 29. TaskGroup with Results

import asyncio


async def compute(n):
    await asyncio.sleep(0.1 * n)
    return n * n


async def main():
    results = []
    async with asyncio.TaskGroup() as tg:
        tasks = [tg.create_task(compute(i)) for i in range(5)]
    results = [t.result() for t in tasks]
    print(results)


await main()

[0, 1, 4, 9, 16]


In [101]:
# 30. TaskGroup Exception Propagation

import asyncio


async def bad_task():
    await asyncio.sleep(0.1)
    raise RuntimeError("Something went wrong")


async def good_task():
    await asyncio.sleep(0.5)
    return "success"


async def main():
    try:
        async with asyncio.TaskGroup() as tg:
            tg.create_task(bad_task())
            tg.create_task(good_task())
    except ExceptionGroup as eg:
        for exc in eg.exceptions:
            print(f"Caught: {exc}")


await main()

Caught: Something went wrong


In [102]:
# 31. Timeout Context Manager

import asyncio


async def long_operation():
    await asyncio.sleep(5)
    return "done"


async def main():
    async with asyncio.timeout(1):
        await long_operation()


await main()

TimeoutError: 

In [104]:
# 32. timeout_at (Absolute Deadline)

import asyncio
import time


async def work():
    await asyncio.sleep(10)
    return "done"


async def main():
    # Monotonic clock, cannot go backward.
    deadline = time.monotonic() + 1.0
    try:
        async with asyncio.timeout_at(deadline):
            await work()
    except asyncio.TimeoutError:
        print("Deadline exceeded")


await main()

Deadline exceeded


In [108]:
# 33. Current Task Info

import asyncio


async def show_task_info():
    task = asyncio.current_task()
    print(f"Task name: {task.get_name()}")
    print(f"Task done: {task.done()}")
    print(f"Task cancelled: {task.cancelled()}")


async def main():
    task = asyncio.create_task(show_task_info(), name="info-task")
    await task


await main()

Task name: info-task
Task done: False
Task cancelled: False


In [111]:
# 34. All Tasks

import asyncio


async def background(i):
    await asyncio.sleep(0.5)
    return i


async def main():
    tasks = [asyncio.create_task(background(i)) for i in range(5)]

    # Show all running tasks
    all_tasks = asyncio.all_tasks()
    print(f"Total tasks: {len(all_tasks)}")

    await asyncio.gather(*tasks)
    print(f"After completion: {len(asyncio.all_tasks())}")


await main()

Total tasks: 7
After completion: 2


In [113]:
# 35. sleep with Zero (Yield Control)

import asyncio


async def cooperative_task(name):
    for i in range(3):
        print(f"{name}: {i}")
        await asyncio.sleep(0)  # Just yield, don't actually sleep


async def main():
    await asyncio.gather(cooperative_task("A"), cooperative_task("B"))


await main()

A: 0
B: 0
A: 1
B: 1
A: 2
B: 2


In [116]:
# 36. gather with kwargs

import asyncio


async def fetch(url, timeout=10):
    await asyncio.sleep(0.1)
    return f"fetched {url} with timeout {timeout}"


async def main():
    results = await asyncio.gather(
        fetch("url1", timeout=5),
        fetch("url2", timeout=20),
    )
    print(results)


await main()

['fetched url1 with timeout 5', 'fetched url2 with timeout 20']


In [1]:
# 37. Dynamic Task Pool

import asyncio


async def worker(worker_id, queue):
    while True:
        item = await queue.get()
        print(f"Worker {worker_id} processing {item}")
        await asyncio.sleep(0.1)
        queue.task_done()


async def main():
    queue = asyncio.Queue()
    workers = [asyncio.create_task(worker(i, queue)) for i in range(3)]

    for item in range(10):
        await queue.put(item)

    await queue.join()
    for w in workers:
        w.cancel()


await main()

Worker 0 processing 0
Worker 1 processing 1
Worker 2 processing 2
Worker 0 processing 3
Worker 1 processing 4
Worker 2 processing 5
Worker 0 processing 6
Worker 1 processing 7
Worker 2 processing 8
Worker 0 processing 9


In [3]:
# 38. Producer-Consumer with Multiple Consumers

import asyncio


async def producer(queue, n):
    for i in range(n):
        await asyncio.sleep(0.05)
        await queue.put(i)
        print(f"Produced: {i}")


async def consumer(consumer_id, queue):
    while True:
        item = await queue.get()
        await asyncio.sleep(0.1)
        print(f"Consumer {consumer_id}: {item}")
        queue.task_done()


async def main():
    queue = asyncio.Queue()
    producers = [asyncio.create_task(producer(queue, 6))]
    consumers = [asyncio.create_task(consumer(i, queue)) for i in range(2)]

    await asyncio.gather(*producers)
    await queue.join()

    for c in consumers:
        c.cancel()


await main()

Produced: 0
Produced: 1
Consumer 0: 0
Produced: 2
Consumer 1: 1
Produced: 3
Consumer 0: 2
Produced: 4
Consumer 1: 3
Produced: 5
Consumer 0: 4
Consumer 1: 5


In [6]:
# 39. Async ContextVar

import asyncio
from contextvars import ContextVar

request_id: ContextVar[str] = ContextVar("request_id", default="unknown")


async def handle_request(rid):
    request_id.set(rid)
    await asyncio.gather(log_request(), process_request())


async def log_request():
    await asyncio.sleep(0.1)
    print(f"Loggiing request: {request_id.get()}")


async def process_request():
    await asyncio.sleep(0.05)
    print(f"Processing request: {request_id.get()}")


async def main():
    await asyncio.gather(handle_request("req-001"), handle_request("req_002"))


await main()

Processing request: req-001
Processing request: req_002
Loggiing request: req-001
Loggiing request: req_002


In [7]:
# 40. To Thread (CPU-bound in ThreadPool)

import asyncio


def cpu_bound(n):
    return sum(i * i for i in range(n))


async def main():
    loop = asyncio.get_running_loop()
    result = await loop.run_in_executor(None, cpu_bound, 10_000_000)
    print(f"Result: {result}")


await main()

Result: 333333283333335000000


In [8]:
# 41. Customer Executor

import asyncio
from concurrent.futures import ThreadPoolExecutor

executor = ThreadPoolExecutor(max_workers=2)

In [13]:
# 42. Custom Executor

import asyncio
from concurrent.futures import ThreadPoolExecutor

executor = ThreadPoolExecutor(max_workers=2)


def heavy_work(n):
    import time

    time.sleep(0.5)
    return n**2


async def main():
    loop = asyncio.get_running_loop()
    results = await asyncio.gather(
        loop.run_in_executor(executor, heavy_work, 10),
        loop.run_in_executor(executor, heavy_work, 20),
        loop.run_in_executor(executor, heavy_work, 30),
    )
    print(results)
    executor.shutdown(wait=False)


await main()

[100, 400, 900]


In [20]:
# 43. asyncio.run with debug

import asyncio


async def main():
    print(f"Debug mode: {asyncio.get_running_loop().get_debug()}")


await main()
# asyncio.run(main(), debug=True)

Debug mode: False


In [23]:
# 44. Slow Callback Detection

import asyncio
import logging

logging.basicConfig(level=logging.WARNING)


async def slow_callback():
    import time

    time.sleep(0.1)  # Blocking


async def main():
    loop = asyncio.get_running_loop()
    loop.slow_callback_during = 0.05
    await slow_callback()


await main()
# asyncio.run(main(), debug=True)

In [24]:
# 45. Subprocess with asyncio

import asyncio


async def run_command(cmd):
    proc = await asyncio.create_subprocess_shell(
        cmd, stdout=asyncio.subprocess.PIPE, stderr=asyncio.subprocess.PIPE
    )
    stdout, stderr = await proc.communicate()
    return stdout.decode().strip(), stderr.decode().strip(), proc.returncode


async def main():
    out, err, code = await run_command("echo 'Hello from subprocess'")
    print(f"Output: {out}")
    print(f"Return code: {code}")


await main()

Output: Hello from subprocess
Return code: 0


In [26]:
# 46 Streming subprocess output

import asyncio


async def stream_output(cmd):
    proc = await asyncio.create_subprocess_shell(
        cmd, stdout=asyncio.subprocess.PIPE, stderr=asyncio.subprocess.STDOUT
    )

    async for line in proc.stdout:
        print(line.decode().strip(), end="\n")

    await proc.wait()
    return proc.returncode


async def main():
    code = await stream_output("echo -e 'line\\nline2\\nline3'")
    print(f"Exit code:{code}")


await main()

-e line
line2
line3
Exit code:0


In [30]:
# 47 Streaming Subprocess Output

import asyncio


async def stream_output(cmd):
    proc = await asyncio.create_subprocess_shell(
        cmd, stdout=asyncio.subprocess.PIPE, stderr=asyncio.subprocess.STDOUT
    )

    async for line in proc.stdout:
        print(line.decode().strip(), end="\n")

    await proc.wait()
    return proc.returncode


async def main():
    code = await stream_output("echo -e 'line1\\nline2\\nline3'")
    print(f"Exit code: {code}")


await main()

-e line1
line2
line3
Exit code: 0


In [3]:
# 47. TCP Echo Server

import asyncio


async def handle_client(reader, writer):
    data = await reader.read(100)
    message = data.decode()
    addr = writer.get_extra_info("peername")
    print(f"Received from {addr}: {message}")

    writer.write(data)
    await write.drain()
    writer.close()
    await writer.wait_closed()


async def main():
    server = await asyncio.start_server(handle_client, "127.0.0.1", 8888)
    addr = server.sockets[0].getsockname()
    print(f"Serving on {addr}")

    async with server:
        await server.serve_forever()


await main()

OSError: [Errno 48] error while attempting to bind on address ('127.0.0.1', 8888): [errno 48] address already in use

In [ ]:
# 48. TCP client

import asyncio


async def tcp_client(message):
    reader, writer = await asyncio.open_connection("127.0.0.1", 8888)

    print(f"Sending: {message}")
    writer.write(message.encode())
    await writer.drain()

    data = await reader.read(100)
    print(f"Received: {data.decode()}")

    writer.close()
    await writer.wait_closed()


async def main():
    await tcp_client("Hello Server!")


await main()

Sending: Hello Server!


In [6]:
# 49. UDP Server and Client

import asyncio


async def udp_server():
    transport, protocol = await asyncio.get_running_loop().create_datagram_endpoint(
        lambda: asyncio.DatagramProtocol(), local_addr=("127.0.0.1", 9999)
    )

    class Handler(asyncio.DatagramProtocol):
        def datagram_received(self, data, addr):
            print("fRecived: {data.decode()} from {addr}")
            self.transport.sendto(data, addr)

    transport.close()
    transport, protocol = await asyncio.get_running_loop().create_datagram_endpoint(
        lambda: Handler(), local_addr=("127.0.0.1", 9999)
    )
    return transport


async def udp_client():
    loop = asyncio.get_running_loop()
    transport, protocol = await loop.create_datagram_endpoint(
        lambda: asycio.DatagramProtocol(), remote_adr=("127.0.0.1", 9999)
    )

    transport.sendto(b"Hello UDP!")
    await asyncio.sleep(0.1)
    transport.close()


async def main():
    server = await udp_server()
    await asyncio.sleep(0.1)
    await udp_client()
    await asyncio.sleep(0.1)
    server.close()


await main()

OSError: [Errno 48] Address already in use

In [2]:
# 50. Retry Pattern with Exponential Backoff

import asyncio
import random


async def unreliable_api():
    if random.random() < 0.7:
        raise ConnectionError("Network error")
    return {"status": "ok"}


async def retry(coro, max_retries=5, base_delay=0.1):
    for attemp in range(max_retries):
        try:
            return await coro()
        except Exception as e:
            if attemp == max_retries - 1:
                raise
            delay = base_delay * (2**attemp) + random.uniform(0, 0.1)
            print(f"Attemp {attemp + 1} failed: {e}. Retrying in {delay:.2f}s")
            await asyncio.sleep(delay)


async def main():
    result = await retry(unreliable_api)
    print(f"Success: {result}")


await main()

Attemp 1 failed: Network error. Retrying in 0.17s
Attemp 2 failed: Network error. Retrying in 0.22s
Attemp 3 failed: Network error. Retrying in 0.41s
Success: {'status': 'ok'}


In [9]:
# 51. Complete Async Application Pattern

import asyncio
from dataclasses import dataclass


@dataclass
class Result:
    url: str
    status: int
    data: str


class AsyncHttpClient:
    def __init__(self, max_concurrent: int = 10):
        self.semaphore = asyncio.Semaphore(max_concurrent)

    async def get(self, url: str) -> Result:
        async with self.semaphore:
            await asyncio.sleep(random.random())  # simulate network
            return Result(url=url, status=200, data=f"data from {url}")


async def process_url(client: AsyncHttpClient, url: str) -> Result:
    print(url)
    return await client.get(url)


async def main():
    urls = [f"http://api.example.com/{i}" for i in range(20)]
    client = AsyncHttpClient(max_concurrent=5)

    async with asyncio.TaskGroup() as tg:
        tasks = [tg.create_task(process_url(client, url)) for url in urls]

    results = [t.result() for t in tasks]
    print(f"Fetched {len(results)} URLs")


await main()

http://api.example.com/0
http://api.example.com/1
http://api.example.com/2
http://api.example.com/3
http://api.example.com/4
http://api.example.com/5
http://api.example.com/6
http://api.example.com/7
http://api.example.com/8
http://api.example.com/9
http://api.example.com/10
http://api.example.com/11
http://api.example.com/12
http://api.example.com/13
http://api.example.com/14
http://api.example.com/15
http://api.example.com/16
http://api.example.com/17
http://api.example.com/18
http://api.example.com/19
Fetched 20 URLs
